# Custom functions and layers

The layers of *nnodely* cover a lot, but every project has a piece of its own:
a friction law, a saturation, a sub-network used in many places. There are four
ways to add one, from the lightest to the most complete:

| Way | What it is | Weights | Saved and exported |
|---|---|---|---|
| a Python function | a function of streams, built from existing layers | the layers' | yes |
| a `Modely` block | a model, called on streams inside another | its own | yes |
| `KerasFunction` | any `keras.layers.Layer`, called on streams | Keras weights | yes, once registered |
| a `Layer` subclass | a new *nnodely* layer, with a Keras implementation | as you define them | yes, once registered |

This tutorial builds one of each, and shows when to choose which.

In [1]:
import os

# Pick the Keras backend before nnodely (or Keras) is imported:
# "tensorflow" (the default), "torch" or "jax".
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import tempfile
from pathlib import Path

import keras
import numpy as np

from nnodely import (
    DataLoader,
    Input,
    KerasFunction,
    Linear,
    Modely,
    Output,
    ReLU,
    set_seed,
)
from nnodely.core.layer import Layer

set_seed(0)
workdir = Path(tempfile.mkdtemp(prefix="nnodely_custom_"))


def run(model, **data):
    results = model({k: np.asarray(v, "float32") for k, v in data.items()})
    return {k: keras.ops.convert_to_numpy(v) for k, v in results.items()}

## A Python function of streams

The lightest way: a function that takes streams and returns a stream, built from
existing layers and operators. Nothing is registered: the model is made of the
layers inside.

**Where the layers are created decides the weights.** A layer created *inside*
the function is a new layer - new weights - at every call. To apply the same
weights in several places, create the layers once, outside, and call them inside.

In [2]:
def mlp(stream, hidden=8):
    """A small network: new weights at every call."""
    return Linear(features=1)(ReLU()(Linear(features=hidden)(stream)))


class SharedMLP:
    """The same network, its layers created once: one set of weights."""

    def __init__(self, hidden=8):
        self.inner, self.outer = Linear(features=hidden), Linear(features=1)

    def __call__(self, stream):
        return self.outer(ReLU()(self.inner(stream)))


a, b = Input("a", dim=3, tw=1), Input("b", dim=3, tw=1)
separate = Modely(
    "separate", inputs=[a, b], outputs=[Output("sum", mlp(a) + mlp(b))]
).build()
shared_net = SharedMLP()
shared = Modely(
    "shared", inputs=[a, b], outputs=[Output("sum", shared_net(a) + shared_net(b))]
).build()
print("separate weights:", len(separate.parameters), "parameters")
print("shared weights:  ", len(shared.parameters), "parameters")

separate weights: 8 parameters
shared weights:   4 parameters


## A `Modely` block

A model is itself a reusable piece: declared and built once, called on streams
inside another model, with its weights shared between every call. It is the way
to package a sub-system trained on its own, frozen or not, saved in a folder of
its own with the model that uses it - see [Model design](01_model_design.ipynb)
and [Save, load and export](05_save_load_export.ipynb).

```python
smoother = Modely("smoother", inputs=[raw], outputs=[Output("smooth", Fir(features=1)([raw]))]).build()
model = Modely("model", inputs=[x], outputs=[Output("y", smoother([x]) * 2)])
```

## `KerasFunction`: any Keras layer

When the computation is not made of *nnodely* layers - a custom formula, a
layer from a Keras library - write it as a `keras.layers.Layer` and wrap it in a
`KerasFunction`. The streams arrive in `call` as tensors laid out
`(batch, *S, T, *D)`, features on the last axis, one argument per stream; the
shape of the result is read off what `call` returns.

A dead zone - zero for small inputs, linear beyond a width the training finds:

In [3]:
@keras.saving.register_keras_serializable(package="tutorial")
class DeadZone(keras.layers.Layer):
    """Zero inside [-width, width], linear outside it; the width is learned."""

    def build(self, input_shape):
        self.width = self.add_weight(
            shape=(), initializer=keras.initializers.Constant(0.5), name="width"
        )

    def call(self, x):
        return keras.ops.sign(x) * keras.ops.relu(keras.ops.abs(x) - self.width)


dead_zone = DeadZone()  # the instance: its weight is shared wherever it is used
u = Input("u", tw=1)
y = Output("y", KerasFunction(dead_zone)([u]))
actuator = Modely("actuator", inputs=[u], outputs=[y])
actuator.minimize("fit", y, Input("y_meas", tw=1))
actuator.build()

u_data = np.random.default_rng(0).uniform(-2.0, 2.0, 2000)
y_data = np.sign(u_data) * np.maximum(np.abs(u_data) - 0.3, 0.0)  # the true width: 0.3
actuator.train(
    DataLoader(actuator, source={"u": u_data, "y_meas": y_data}),
    epochs=20,
    batch_size=32,
    lr=1e-2,
    printer=None,
)
print("learned width:", round(float(keras.ops.convert_to_numpy(dead_zone.width)), 4))

learned width: 0.3


Its weights are **Keras** weights: trained with the model, but not
`Parameter`s - `model.parameters` and `model.freeze()` do not reach them; set
`dead_zone.trainable = False` to freeze it.

A layer whose `call` takes several arguments takes as many streams, in order:

In [4]:
@keras.saving.register_keras_serializable(package="tutorial")
class Gate(keras.layers.Layer):
    """The signal, let through as much as the sigmoid of the gate."""

    def call(self, signal, gate):
        return signal * keras.ops.sigmoid(gate)


s, g = Input("s", dim=2, tw=3), Input("g", tw=3)
gated = KerasFunction(Gate)([s, g])  # a class: made here, without arguments
print(gated.shape)

Shape(S=(), T=3, D=(2,))


`@keras.saving.register_keras_serializable()` lets Keras save and load the
layer: a model holding it is saved and exported as any other - in another
session, define (or import) the class before loading.

## A new *nnodely* layer

A `Layer` subclass makes a layer of the same kind as `Sin` or `Fir`: configured
once, called on streams, listed with its own class name, saved by its
arguments. It is two classes:

- a **Keras layer** - the implementation, which computes on tensors;
- a ***nnodely* `Layer`** - the declaration: it takes the settings, passes them
  to `super().__init__` (they are what is saved, and what the layer is made
  again from), and returns the Keras layer from `build_layer()`.

A soft saturation, $y = L \tanh(x / L)$, with the limit `L` as a setting:

In [5]:
@keras.saving.register_keras_serializable(package="tutorial")
class SoftClipImpl(keras.layers.Layer):
    def __init__(self, limit=1.0, **kwargs):
        super().__init__(**kwargs)
        self.limit = float(limit)

    def call(self, x):
        return self.limit * keras.ops.tanh(x / self.limit)

    def get_config(self):
        return {**super().get_config(), "limit": self.limit}


class SoftClip(Layer):
    """limit * tanh(x / limit): close to x for small values, at most limit."""

    def __init__(self, limit=1.0, name=None):
        self._limit = float(limit)
        super().__init__(name=name, limit=self._limit)  # every setting goes here

    def build_layer(self):
        return SoftClipImpl(limit=self._limit, name=self.name)


x = Input("x", dim=2, tw=4)
clipped = SoftClip(limit=2.0)(x)
print(clipped.name, clipped.shape)

soft = Modely("soft", inputs=[x], outputs=[Output("clipped", clipped)]).build()
values = np.array([[[0.1, -0.1], [1.0, -1.0], [3.0, -3.0], [10.0, -10.0]]])
print(run(soft, x=values)["clipped"][0].round(3))

SoftClip16 Shape(S=(), T=4, D=(2,))
[[ 0.1   -0.1  ]
 [ 0.924 -0.924]
 [ 1.81  -1.81 ]
 [ 2.    -2.   ]]


The output shape is inferred by running the Keras layer once on zeros: here it
keeps the shape of its input. A layer that changes it needs nothing more - the
result's axes are read as `(batch, *S, T, *D)`:

In [6]:
@keras.saving.register_keras_serializable(package="tutorial")
class EnergyImpl(keras.layers.Layer):
    def call(self, x):
        return keras.ops.sum(x * x, axis=-2, keepdims=True)  # over the window


class Energy(Layer):
    """The energy of each feature over the window: one sample."""

    def __init__(self, name=None):
        super().__init__(name=name)

    def build_layer(self):
        return EnergyImpl(name=self.name)


print(Energy()(x).shape)  # the window of 4 summed into one sample

Shape(S=(), T=1, D=(2,))


Override `output_shape(self, *inputs)` only when the shape cannot be read that
way - a layer that adds or removes sequence axes, or whose shape depends on the
values. Use `keras.ops` only, never the functions of one backend, so that the
layer runs on all of them.

A model holding the new layer saves, loads and exports as any other:

In [7]:
soft.save(workdir / "soft")
restored = Modely.load(workdir / "soft")
print(np.allclose(run(restored, x=values)["clipped"], run(soft, x=values)["clipped"]))
soft.export_keras(workdir)
print(sorted(path.name for path in workdir.glob("*.keras")))

True
['soft.keras']


### Weights as Parameters

`KerasFunction` keeps its weights in Keras. A `Layer` subclass can instead expose
them as `Parameter`s - given, shared, frozen and listed like those of `Fir` and
`Linear` - by subclassing `_ParameterWeights` from `nnodely.layers.parameter`,
as `Fir`, `Linear`, `PReLU` and `BatchNorm` do: the source of those layers is the
reference.

## Which one?

- Built from existing layers? A **function** - or a `Modely` block, to train,
  freeze or save it as a unit.
- A formula of its own, or a Keras layer you already have? **`KerasFunction`**.
- A layer to reuse across projects, with settings and a name of its own? A
  **`Layer` subclass**.

## Summary

| Task | Code |
|---|---|
| A reusable piece of graph | `def f(stream): return Linear(...)(...)`; layers created once to share weights |
| A reusable model | `block = Modely(...).build()`, then `block([stream])` |
| Any Keras layer | `KerasFunction(MyKerasLayer)([streams])` or `KerasFunction(instance)` |
| Several arguments | `call(self, x, y)` takes `[x_stream, y_stream]` |
| A new layer | `class MyLayer(Layer)`: settings to `super().__init__`, `build_layer()` returns the Keras layer |
| Save it | `@keras.saving.register_keras_serializable()` on the Keras layer |

Next: [EquationLearner](13_equation_learner.ipynb).